# Laser Source Gradient Analysis

Calibration parameter sweep analysis using the `gradient_analysis` library.

In [ ]:
import sys
sys.path.append('../')

from lucid.geometry import generate_detector
from lucid.losses import WC_smooth_loss
from lucid.simulation import setup_event_simulator
from lucid.detector_params import laser_source, load_detector_params
from lucid.gradient_analysis import (
    SweepParam, sweep_1d, sweep_2d, plot_sweep_1d, plot_sweep_2d,
)

import jax
import jax.numpy as jnp
from jax import jit, value_and_grad

In [ ]:
default_json_filename = '../config/SK_geom_config.json'
detector = generate_detector(default_json_filename)
detector_points = jnp.array(detector.all_points)
NUM_DETECTORS = len(detector_points)

Nphot = 500_000
K = 8
Nphot_True = 15_000_000
K_True = 12

TRUE_PARAMS = load_detector_params('../config/SK_physics_config.json', num_sensors=NUM_DETECTORS)

source = laser_source(position=[0.0, 0.0, detector.H / 2 - 0.1], intensity=100_000_000)

simulate_event = setup_event_simulator(
    default_json_filename, Nphot, temperature=None, K=K,
    is_data=False, is_calibration=True,
)
simulate_data = setup_event_simulator(
    default_json_filename, Nphot_True, temperature=None, K=K_True,
    is_data=False, is_calibration=True,
    default_detector_params=TRUE_PARAMS,
)

key = jax.random.PRNGKey(42)
key_source, key_data = jax.random.split(key)
true_data = jax.lax.stop_gradient(simulate_data(source, key_data))

In [ ]:
calib_sweeps = [
    SweepParam('Scatter Length',    'scatter_length',         half_width=20.0,  unit='m', min_val=0.001, grad_scale=100.0),
    SweepParam('Wall Reflection',   'wall_reflection_rate',   half_width=0.15,  min_val=0.0, max_val=1.0, grad_scale=0.1),
    SweepParam('Sensor Reflection', 'sensor_reflection_rate', half_width=0.08,  min_val=0.0, max_val=1.0, grad_scale=0.1),
    SweepParam('Absorption Length', 'absorption_length',      half_width=80.0,  unit='m', min_val=25.0, grad_scale=100.0), # Add min because any smaller and it blows up
]

In [ ]:
@jit
def loss_and_grad_fn(detector_params):
    def loss_fn(dp):
        simulated_data = simulate_event(source, dp, key_source)
        return WC_smooth_loss(
            detector_points, *true_data, *simulated_data,
            lambda_poisson=1.0, lambda_time=0.0, tau=0.5,
        )
    return value_and_grad(loss_fn)(detector_params)

_ = loss_and_grad_fn(TRUE_PARAMS)  # warmup

In [ ]:
results_1d = sweep_1d(loss_and_grad_fn, TRUE_PARAMS, calib_sweeps)

In [ ]:
plot_sweep_1d(results_1d, title='Calibration Parameter 1D Sweeps', save_path='figures/laser_1d_sweeps_v3.png')

In [ ]:
pairs_2d = [
    (calib_sweeps[0], calib_sweeps[3], 31),  # Scatter Length x Absorption Length
    (calib_sweeps[1], calib_sweeps[2], 41),  # Wall Reflection x Sensor Reflection
    (calib_sweeps[0], calib_sweeps[1], 41),  # Scatter Length x Wall Reflection
    (calib_sweeps[0], calib_sweeps[2], 41),  # Scatter Length x Sensor Reflection
]
results_2d = [sweep_2d(loss_and_grad_fn, TRUE_PARAMS, px, py, num_points=n) for px, py, n in pairs_2d]

In [ ]:
plot_sweep_2d(results_2d, title='Calibration Parameter 2D Surfaces', save_path='figures/laser_2d_surfaces_v3.png')